# boundary_qef GPU Compare

JIT compile current `intersect_qef`, the three boundary QEF variants, a shared temporary CUDA boundary extractor, and a CPU `boundry_qef` wrapper. `intersect_qef` is used only to prepare active voxels and brick lookup tensors. The three GPU boundary functions expose the same Python signature and compute `boundaries` internally before calling their respective QEF path.

In [1]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))
os.environ['PATH'] = f"{CONDA_PREFIX / 'bin'}:{os.environ.get('PATH', '')}"
os.environ['CUDA_HOME'] = str(CONDA_PREFIX)
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

import torch
import torch.utils.cpp_extension as cpp_extension
from torch.utils.cpp_extension import load

cpp_extension.CUDA_HOME = str(CONDA_PREFIX)
torch.set_grad_enabled(False)

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '512'))
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))
BOUNDARY_CHUNK_STEPS = int(os.environ.get('FDG_BOUNDARY_CHUNK_STEPS', '64'))
BOUNDARY_WEIGHT = float(os.environ.get('FDG_BOUNDARY_WEIGHT', '1.0'))
SUBDIVIDE_AREA_THRESHOLD = float(os.environ.get('FDG_SUBDIVIDE_AREA_THRESHOLD', str(1.0 / (GRID_SIZE * GRID_SIZE))))
SUBDIVIDE_MAX_ITERS = int(os.environ.get('FDG_SUBDIVIDE_MAX_ITERS', '12'))
WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
RUN_CPU_SUBDIVIDED = os.environ.get('FDG_RUN_CPU_SUBDIVIDED', '1') != '0'

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('CONDA_PREFIX:', CONDA_PREFIX)
print('nvcc:', CONDA_PREFIX / 'bin/nvcc')
print('gcc:', os.environ['CC'])
print('g++:', os.environ['CXX'])
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)
print('BOUNDARY_WEIGHT:', BOUNDARY_WEIGHT, 'BOUNDARY_CHUNK_STEPS:', BOUNDARY_CHUNK_STEPS)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)
print('WARM_REPEATS:', WARM_REPEATS, 'RUN_CPU_SUBDIVIDED:', RUN_CPU_SUBDIVIDED)


repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
CONDA_PREFIX: /home/quanta/.conda/envs/symm-enforce
nvcc: /home/quanta/.conda/envs/symm-enforce/bin/nvcc
gcc: /home/quanta/.conda/envs/symm-enforce/bin/gcc
g++: /home/quanta/.conda/envs/symm-enforce/bin/g++
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000
BOUNDARY_WEIGHT: 1.0 BOUNDARY_CHUNK_STEPS: 64
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12
WARM_REPEATS: 10 RUN_CPU_SUBDIVIDED: True


## JIT Extension

The temporary binding includes a CPU `boundry_qef` wrapper and two boundary-extracting GPU wrappers for old/current. The ref path is not wrapped with a synthetic implementation: it directly calls the real `o_voxel::fdg::boundary_qef_ref(...)`, which performs its own boundary extraction internally.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_boundary_qef_jit_'))
binding_path = build_dir / 'binding.cpp'
temp_cuda_path = build_dir / 'boundary_temp.cu'
torch_build_dir = build_dir / 'torch_ext'
torch_build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = r"""
#include <torch/extension.h>
#include <Eigen/Dense>
#include <unordered_map>
#include <vector>
#include <cstdint>
#include "__ROOT__/src/convert/flexible_dual_grid.cpp"
#include "__ROOT__/src/convert/mesh_to_flexible_dual_grid_gpu/api.h"

namespace py = pybind11;

torch::Tensor extract_boundaries_cuda(
    const torch::Tensor& vertices,
    const torch::Tensor& faces);


namespace {

uint64_t pack_edge_key_cpu(int32_t a, int32_t b) {
    return (static_cast<uint64_t>(static_cast<uint32_t>(a)) << 32) |
           static_cast<uint32_t>(b);
}

torch::Tensor boundary_qef_cpu_only(
    const torch::Tensor& vertices,
    const torch::Tensor& faces,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    float boundary_weight,
    const torch::Tensor& voxels) {
    const int64_t F = faces.size(0);
    const int64_t N = voxels.size(0);
    const float* v_ptr = vertices.data_ptr<float>();
    const int32_t* f_ptr = faces.data_ptr<int32_t>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vox_ptr = voxels.data_ptr<int32_t>();

    std::unordered_map<uint64_t, int32_t> edge_count;
    edge_count.reserve(static_cast<size_t>(F) * 4);
    for (int64_t f = 0; f < F; ++f) {
        const int32_t fv[3] = {f_ptr[3 * f + 0], f_ptr[3 * f + 1], f_ptr[3 * f + 2]};
        for (int e = 0; e < 3; ++e) {
            int32_t a = fv[e];
            int32_t b = fv[(e + 1) % 3];
            if (a > b) {
                const int32_t t = a;
                a = b;
                b = t;
            }
            edge_count[pack_edge_key_cpu(a, b)] += 1;
        }
    }

    std::vector<Eigen::Vector3f> boundaries;
    boundaries.reserve(edge_count.size() * 2);
    for (const auto& kv : edge_count) {
        if (kv.second != 1)
            continue;
        const int32_t a = static_cast<int32_t>(kv.first >> 32);
        const int32_t b = static_cast<int32_t>(kv.first & 0xffffffffu);
        boundaries.emplace_back(v_ptr[3 * static_cast<int64_t>(a) + 0],
                                v_ptr[3 * static_cast<int64_t>(a) + 1],
                                v_ptr[3 * static_cast<int64_t>(a) + 2]);
        boundaries.emplace_back(v_ptr[3 * static_cast<int64_t>(b) + 0],
                                v_ptr[3 * static_cast<int64_t>(b) + 1],
                                v_ptr[3 * static_cast<int64_t>(b) + 2]);
    }

    std::unordered_map<VoxelCoord, size_t> hash_table;
    hash_table.reserve(static_cast<size_t>(N) * 2);
    for (int64_t i = 0; i < N; ++i) {
        hash_table[VoxelCoord{vox_ptr[3 * i + 0], vox_ptr[3 * i + 1], vox_ptr[3 * i + 2]}] = static_cast<size_t>(i);
    }

    std::vector<Eigen::Matrix4f> qefs(static_cast<size_t>(N), Eigen::Matrix4f::Zero());
    ::boundry_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        boundaries,
        boundary_weight,
        hash_table,
        qefs);

    auto out = torch::empty({N, 10}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    float* out_ptr = out.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {
        const Eigen::Matrix4f& Q = qefs[static_cast<size_t>(i)];
        out_ptr[10 * i + 0] = Q(0, 0);
        out_ptr[10 * i + 1] = Q(0, 1);
        out_ptr[10 * i + 2] = Q(0, 2);
        out_ptr[10 * i + 3] = Q(0, 3);
        out_ptr[10 * i + 4] = Q(1, 1);
        out_ptr[10 * i + 5] = Q(1, 2);
        out_ptr[10 * i + 6] = Q(1, 3);
        out_ptr[10 * i + 7] = Q(2, 2);
        out_ptr[10 * i + 8] = Q(2, 3);
        out_ptr[10 * i + 9] = Q(3, 3);
    }
    return out;
}

torch::Tensor boundary_qef_old_full(
    const torch::Tensor& vertices,
    const torch::Tensor& faces,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    float boundary_weight,
    const torch::Tensor& voxels,
    const torch::Tensor& brick_hash_keys,
    const torch::Tensor& brick_hash_vals,
    const torch::Tensor& brick_bits,
    const torch::Tensor& brick_base,
    int chunk_steps) {
    (void)brick_hash_keys;
    (void)brick_hash_vals;
    (void)brick_bits;
    (void)brick_base;
    auto boundaries = extract_boundaries_cuda(vertices, faces);
    return o_voxel::fdg::boundary_qef_old(boundaries, voxel_size, grid_range, boundary_weight, voxels, chunk_steps);
}


torch::Tensor boundary_qef_full(
    const torch::Tensor& vertices,
    const torch::Tensor& faces,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    float boundary_weight,
    const torch::Tensor& voxels,
    const torch::Tensor& brick_hash_keys,
    const torch::Tensor& brick_hash_vals,
    const torch::Tensor& brick_bits,
    const torch::Tensor& brick_base,
    int chunk_steps) {
    (void)chunk_steps;
    auto boundaries = extract_boundaries_cuda(vertices, faces);
    return o_voxel::fdg::boundary_qef(
        boundaries, voxel_size, grid_range, boundary_weight, voxels,
        brick_hash_keys, brick_hash_vals, brick_bits, brick_base);
}

} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("extract_boundaries", &extract_boundaries_cuda);
    m.def("boundary_qef_cpu", &boundary_qef_cpu_only);
    m.def("boundary_qef_old_full", &boundary_qef_old_full);
    m.def("boundary_qef_ref", &o_voxel::fdg::boundary_qef_ref);
    m.def("boundary_qef_full", &boundary_qef_full);
}
""".replace('__ROOT__', str(ROOT).replace('\\', '/'))

boundary_temp_cuda = r"""
#include <torch/extension.h>
#include "types.cuh"
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_runtime.h>

#include <cstdint>
#include <limits>

using namespace o_voxel::fdg;

namespace {

constexpr int kThreads = 256;
constexpr uint64_t kEmptyEdgeKey = UINT64_MAX;

__device__ __forceinline__ uint64_t mix64(uint64_t x) {
    x ^= x >> 33;
    x *= 0xff51afd7ed558ccdULL;
    x ^= x >> 33;
    x *= 0xc4ceb9fe1a85ec53ULL;
    x ^= x >> 33;
    return x;
}

__host__ __device__ __forceinline__ uint64_t pack_edge_key(int32_t a, int32_t b) {
    return (static_cast<uint64_t>(static_cast<uint32_t>(a)) << 32) |
           static_cast<uint32_t>(b);
}

__global__ void count_edges_kernel(
    int64_t num_faces,
    const int32_t* __restrict__ faces,
    uint64_t* __restrict__ hash_keys,
    uint32_t* __restrict__ edge_counts,
    uint64_t hash_capacity) {
    const int64_t fid = static_cast<int64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (fid >= num_faces)
        return;

    const int32_t fv[3] = {faces[3 * fid + 0], faces[3 * fid + 1], faces[3 * fid + 2]};
    for (int e = 0; e < 3; ++e) {
        int32_t a = fv[e];
        int32_t b = fv[(e + 1) % 3];
        if (a > b) {
            const int32_t t = a;
            a = b;
            b = t;
        }
        const uint64_t key = pack_edge_key(a, b);
        uint64_t slot = mix64(key) & (hash_capacity - 1);
        for (uint64_t probe = 0; probe < hash_capacity; ++probe) {
            const uint64_t old = atomicCAS(
                reinterpret_cast<unsigned long long*>(hash_keys + slot),
                static_cast<unsigned long long>(kEmptyEdgeKey),
                static_cast<unsigned long long>(key));
            if (old == kEmptyEdgeKey || old == key) {
                atomicAdd(edge_counts + slot, 1u);
                break;
            }
            slot = (slot + 1u) & (hash_capacity - 1);
        }
    }
}

__global__ void count_boundary_edges_kernel(
    uint64_t hash_capacity,
    const uint64_t* __restrict__ hash_keys,
    const uint32_t* __restrict__ edge_counts,
    uint32_t* __restrict__ boundary_count) {
    const uint64_t slot = static_cast<uint64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (slot >= hash_capacity)
        return;
    if (hash_keys[slot] != kEmptyEdgeKey && edge_counts[slot] == 1u)
        atomicAdd(boundary_count, 1u);
}

__global__ void emit_boundaries_kernel(
    uint64_t hash_capacity,
    const uint64_t* __restrict__ hash_keys,
    const uint32_t* __restrict__ edge_counts,
    const float* __restrict__ vertices,
    uint32_t* __restrict__ boundary_count,
    float* __restrict__ boundaries) {
    const uint64_t slot = static_cast<uint64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (slot >= hash_capacity)
        return;
    const uint64_t key = hash_keys[slot];
    if (key == kEmptyEdgeKey || edge_counts[slot] != 1u)
        return;

    const uint32_t out = atomicAdd(boundary_count, 1u);
    const int32_t a = static_cast<int32_t>(key >> 32);
    const int32_t b = static_cast<int32_t>(key & 0xffffffffu);
    boundaries[6 * static_cast<int64_t>(out) + 0] = vertices[3 * static_cast<int64_t>(a) + 0];
    boundaries[6 * static_cast<int64_t>(out) + 1] = vertices[3 * static_cast<int64_t>(a) + 1];
    boundaries[6 * static_cast<int64_t>(out) + 2] = vertices[3 * static_cast<int64_t>(a) + 2];
    boundaries[6 * static_cast<int64_t>(out) + 3] = vertices[3 * static_cast<int64_t>(b) + 0];
    boundaries[6 * static_cast<int64_t>(out) + 4] = vertices[3 * static_cast<int64_t>(b) + 1];
    boundaries[6 * static_cast<int64_t>(out) + 5] = vertices[3 * static_cast<int64_t>(b) + 2];
}


int64_t next_power_of_two_i64(int64_t x) {
    int64_t p = 1;
    while (p < x)
        p <<= 1;
    return p;
}

} // namespace

torch::Tensor extract_boundaries_cuda(
    const torch::Tensor& vertices,
    const torch::Tensor& faces) {
    const c10::cuda::CUDAGuard guard(vertices.device());
    const cudaStream_t stream = at::cuda::getCurrentCUDAStream(vertices.get_device()).stream();
    const torch::Device device = vertices.device();
    const int64_t num_faces = faces.size(0);
    const auto opts_u64 = torch::TensorOptions().dtype(torch::kUInt64).device(device);
    const auto opts_u32 = torch::TensorOptions().dtype(torch::kUInt32).device(device);
    const auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(device);
    if (num_faces == 0)
        return torch::empty({0, 2, 3}, opts_f32);

    const int64_t num_edges = num_faces * 3;
    const int64_t hash_capacity_i64 = next_power_of_two_i64(std::max<int64_t>(2, num_edges * 2));
    auto hash_keys = torch::empty({hash_capacity_i64}, opts_u64);
    auto edge_counts = torch::zeros({hash_capacity_i64}, opts_u32);
    C10_CUDA_CHECK(cudaMemsetAsync(hash_keys.data_ptr<uint64_t>(), 0xff, hash_capacity_i64 * sizeof(uint64_t), stream));

    int blocks = static_cast<int>((num_faces + kThreads - 1) / kThreads);
    count_edges_kernel<<<blocks, kThreads, 0, stream>>>(
        num_faces,
        faces.data_ptr<int32_t>(),
        hash_keys.data_ptr<uint64_t>(),
        edge_counts.data_ptr<uint32_t>(),
        static_cast<uint64_t>(hash_capacity_i64));
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    auto boundary_count_t = torch::zeros({1}, opts_u32);
    blocks = static_cast<int>((hash_capacity_i64 + kThreads - 1) / kThreads);
    count_boundary_edges_kernel<<<blocks, kThreads, 0, stream>>>(
        static_cast<uint64_t>(hash_capacity_i64),
        hash_keys.data_ptr<uint64_t>(),
        edge_counts.data_ptr<uint32_t>(),
        boundary_count_t.data_ptr<uint32_t>());
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    uint32_t boundary_count = 0;
    C10_CUDA_CHECK(cudaMemcpyAsync(&boundary_count, boundary_count_t.data_ptr<uint32_t>(), sizeof(uint32_t), cudaMemcpyDeviceToHost, stream));
    C10_CUDA_CHECK(cudaStreamSynchronize(stream));
    auto boundaries = torch::empty({static_cast<int64_t>(boundary_count), 2, 3}, opts_f32);
    if (boundary_count == 0)
        return boundaries;

    C10_CUDA_CHECK(cudaMemsetAsync(boundary_count_t.data_ptr<uint32_t>(), 0, sizeof(uint32_t), stream));
    emit_boundaries_kernel<<<blocks, kThreads, 0, stream>>>(
        static_cast<uint64_t>(hash_capacity_i64),
        hash_keys.data_ptr<uint64_t>(),
        edge_counts.data_ptr<uint32_t>(),
        vertices.data_ptr<float>(),
        boundary_count_t.data_ptr<uint32_t>(),
        boundaries.data_ptr<float>());
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return boundaries;
}

"""

binding_path.write_text(binding_cpp)
temp_cuda_path.write_text(boundary_temp_cuda)

sources = [
    str(binding_path),
    str(temp_cuda_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef.cu'),
]

ext = load(
    name=f'fdg_boundary_qef_jit_{os.getpid()}',
    sources=sources,
    build_directory=str(torch_build_dir),
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17', '-ccbin', str(CONDA_PREFIX / 'bin/gcc')],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)


Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/fdg_boundary_qef_jit_dtuf9sqv/torch_ext/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_boundary_qef_jit_1545786...
Using envvar MAX_JOBS (32) as the number of workers...


[1/7] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_boundary_qef_jit_1545786 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -s

Loading extension module fdg_boundary_qef_jit_1545786...


## Load Mesh Inputs

The original mesh comes from `notebooks/test.glb`. The subdivided input uses the same GPU subdivision helper as the intersect/face notebooks. Faces are kept as `int32` for the temporary CUDA boundary extractor.

In [3]:
import numpy as np
import trimesh


def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)

        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1)
        p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2)
        p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0)
        p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))

        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]
            fe1 = f2e[m1]
            M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]

            split_col = target[fe1].long().argmax(dim=1)
            c01_m1 = col01[m1]
            c12_m1 = col12[m1]
            c20_m1 = col20[m1]
            is_split_01 = split_col == c01_m1
            is_split_12 = split_col == c12_m1
            is_split_20 = split_col == c20_m1

            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]

            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)

            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)

            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]
            fem = f2e[mm]
            M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]

            c01_m = col01[mm]
            c12_m = col12[mm]
            c20_m = col20[mm]

            eid01 = fem[torch.arange(M2, device=device), c01_m]
            e01_is_split = target[eid01]
            e01_v = torch.where(e01_is_split, e2new[eid01], v0_m)

            eid12 = fem[torch.arange(M2, device=device), c12_m]
            e12_is_split = target[eid12]
            e12_v = torch.where(e12_is_split, e2new[eid12], v1_m)

            eid20 = fem[torch.arange(M2, device=device), c20_m]
            e20_is_split = target[eid20]
            e20_v = torch.where(e20_is_split, e2new[eid20], v2_m)

            mf1 = torch.stack([v0_m, e01_v, e20_v], dim=1)
            mf2 = torch.stack([e01_v, v1_m, e12_v], dim=1)
            mf3 = torch.stack([e20_v, e12_v, v2_m], dim=1)
            mf4 = torch.stack([e01_v, e12_v, e20_v], dim=1)
            out.extend([mf1, mf2, mf3, mf4])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def triangle_mb(triangles: torch.Tensor) -> float:
    return triangles.numel() * triangles.element_size() / 1024**2


def tensor_mb(t: torch.Tensor) -> float:
    return t.numel() * t.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu_long = torch.from_numpy(faces_np).contiguous()
faces_cpu_i32 = faces_cpu_long.to(torch.int32).contiguous()
triangles_cpu = vertices_cpu[faces_cpu_long].reshape(-1, 9).contiguous()
vertices_cuda = vertices_cpu.to(DEVICE, non_blocking=True).contiguous()
faces_cuda_i32 = faces_cpu_i32.to(DEVICE, non_blocking=True).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache()
torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda_long = subdivide_mesh_gpu(
    vertices_cuda,
    faces_cpu_long.to(DEVICE, non_blocking=True).contiguous(),
    area_threshold=SUBDIVIDE_AREA_THRESHOLD,
    max_iters=SUBDIVIDE_MAX_ITERS,
)
sub_faces_cuda_i32 = sub_faces_cuda_long.to(torch.int32).contiguous()
sub_triangles_cuda = sub_vertices_cuda[sub_faces_cuda_long].reshape(-1, 9).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_vertices_cpu = sub_vertices_cuda.detach().cpu().contiguous()
sub_faces_cpu_i32 = sub_faces_cuda_i32.detach().cpu().contiguous()
sub_triangles_cpu = sub_triangles_cuda.detach().cpu().contiguous()

torch.cuda.empty_cache()

INPUT_CASES = {
    'original': {
        'vertices_cpu': vertices_cpu,
        'faces_cpu_i32': faces_cpu_i32,
        'triangles_cpu': triangles_cpu,
        'vertices_cuda': vertices_cuda,
        'faces_cuda_i32': faces_cuda_i32,
        'triangles_cuda': triangles_cuda,
        'vertices_shape': tuple(vertices_cpu.shape),
        'faces_shape': tuple(faces_cpu_i32.shape),
        'triangles': int(triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(triangles_cuda),
        'mesh_mb': tensor_mb(vertices_cuda) + tensor_mb(faces_cuda_i32),
        'preprocess_ms': 0.0,
    },
    'subdivided': {
        'vertices_cpu': sub_vertices_cpu,
        'faces_cpu_i32': sub_faces_cpu_i32,
        'triangles_cpu': sub_triangles_cpu,
        'vertices_cuda': sub_vertices_cuda,
        'faces_cuda_i32': sub_faces_cuda_i32,
        'triangles_cuda': sub_triangles_cuda,
        'vertices_shape': tuple(sub_vertices_cpu.shape),
        'faces_shape': tuple(sub_faces_cpu_i32.shape),
        'triangles': int(sub_triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(sub_triangles_cuda),
        'mesh_mb': tensor_mb(sub_vertices_cuda) + tensor_mb(sub_faces_cuda_i32),
        'preprocess_ms': subdivide_ms,
    },
}

for name, case in INPUT_CASES.items():
    print(
        name,
        'vertices:', case['vertices_shape'],
        'faces:', case['faces_shape'],
        'triangles:', case['triangles'],
        'triangle_mb:', case['triangle_mb'],
        'mesh_mb:', case['mesh_mb'],
        'preprocess_ms:', case['preprocess_ms'],
    )
print('face multiplier:', float(INPUT_CASES['subdivided']['triangles'] / max(INPUT_CASES['original']['triangles'], 1)))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())


building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) triangles: 280333 triangle_mb: 9.624469757080078 mesh_mb: 6.275379180908203 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) triangles: 10304077 triangle_mb: 353.7624092102051 mesh_mb: 159.22097396850586 preprocess_ms: 826.0291740007233
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


## Helpers

GPU functions take CUDA mesh tensors. `voxel_size` and `grid_range` stay on CPU because the current C++ code reads those scalar tensors on host. Boundary QEF outputs are canonicalized by voxel key before comparison, even though all variants should return rows aligned with the shared `intersect_qef.voxels`.

In [4]:
from typing import NamedTuple


class IntersectOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor
    brick_hash_keys: torch.Tensor
    brick_hash_vals: torch.Tensor
    brick_bits: torch.Tensor
    brick_base: torch.Tensor


def as_intersect_output(output):
    assert len(output) == 9, f'expected 9 tensors from intersect_qef, got {len(output)}'
    return IntersectOutput(*output)


def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonical_boundary_qef(voxels: torch.Tensor, qefs: torch.Tensor):
    voxels_cpu = voxels.detach().cpu().contiguous()
    qefs_cpu = qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels_cpu)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels_cpu[order],
        'qefs': qefs_cpu[order],
        'unique': bool(unique),
    }


def compare_keys(base, other):
    a = base['keys']
    b = other['keys']
    equal = torch.equal(a, b)
    if equal:
        return {'occ_equal': True, 'count': int(a.numel())}
    aset = set(a.tolist())
    bset = set(b.tolist())
    return {
        'occ_equal': False,
        'count_a': int(a.numel()),
        'count_b': int(b.numel()),
        'missing': len(aset - bset),
        'extra': len(bset - aset),
        'missing_sample': sorted(list(aset - bset))[:8],
        'extra_sample': sorted(list(bset - aset))[:8],
    }


def boundary_error_metrics(cpu, other):
    keys = compare_keys(cpu, other)
    if not keys['occ_equal']:
        return keys
    diff = (other['qefs'] - cpu['qefs']).float()
    ref = cpu['qefs'].float()
    abs_diff = diff.abs()
    trace_diff = diff[:, [0, 4, 7]].sum(dim=1) if diff.numel() else torch.empty(0)
    return {
        **keys,
        'qef_max_abs': float(abs_diff.max().item()) if diff.numel() else 0.0,
        'qef_rmse': float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0,
        'qef_rel_l2': float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0,
        'trace_max_abs': float(trace_diff.abs().max().item()) if trace_diff.numel() else 0.0,
        'nonzero_cpu': int((cpu['qefs'].abs().sum(dim=1) > 0).sum().item()),
        'nonzero_other': int((other['qefs'].abs().sum(dim=1) > 0).sum().item()),
    }


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def boundary_args(case, inter):
    return (
        case['vertices_cuda'],
        case['faces_cuda_i32'],
        voxel_size_cpu,
        grid_range_cpu,
        BOUNDARY_WEIGHT,
        inter.voxels,
        inter.brick_hash_keys,
        inter.brick_hash_vals,
        inter.brick_bits,
        inter.brick_base,
        BOUNDARY_CHUNK_STEPS,
    )


def boundary_qef_old_py(case, inter):
    return ext.boundary_qef_old_full(*boundary_args(case, inter))


def boundary_qef_ref_py(case, inter):
    return ext.boundary_qef_ref(
        case['vertices_cuda'],
        case['faces_cuda_i32'],
        voxel_size_cpu,
        grid_range_cpu,
        BOUNDARY_WEIGHT,
        inter.voxels,
        inter.brick_hash_keys,
        inter.brick_hash_vals,
        inter.brick_bits,
        inter.brick_base,
    )


def boundary_qef_new_py(case, inter):
    return ext.boundary_qef_full(*boundary_args(case, inter))


def boundary_qef_cpu_py(case, inter):
    return ext.boundary_qef_cpu(
        case['vertices_cpu'],
        case['faces_cpu_i32'],
        voxel_size_cpu,
        grid_range_cpu,
        BOUNDARY_WEIGHT,
        inter.voxels.detach().cpu().contiguous(),
    )


## Build Active Voxel And Brick Inputs

This cell runs only current `intersect_qef`. Its output is the shared active voxel set and brick lookup used by all boundary QEF variants.

In [5]:
torch.cuda.empty_cache()
torch.cuda.synchronize()
INTERSECT_OUTPUTS = {}
intersect_rows = []

for input_name, case in INPUT_CASES.items():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = as_intersect_output(ext.intersect_qef(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES))
    end.record()
    torch.cuda.synchronize()
    INTERSECT_OUTPUTS[input_name] = out
    intersect_rows.append({
        'input': input_name,
        'triangles': case['triangles'],
        'voxels': int(out.voxels.shape[0]),
        'intersect_ms': float(start.elapsed_time(end)),
        'intersect_peak_alloc_mb': float((torch.cuda.max_memory_allocated() - base_alloc) / 1024**2),
        'brick_hash_slots': int(out.brick_hash_keys.numel()),
        'brick_bits_shape': tuple(out.brick_bits.shape),
    })

show_rows(intersect_rows)

boundary_info_rows = []
for input_name, case in INPUT_CASES.items():
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    boundaries = ext.extract_boundaries(case['vertices_cuda'], case['faces_cuda_i32'])
    end.record()
    torch.cuda.synchronize()
    boundary_info_rows.append({
        'input': input_name,
        'faces': int(case['faces_cuda_i32'].shape[0]),
        'boundary_edges': int(boundaries.shape[0]),
        'extract_ms': float(start.elapsed_time(end)),
        'boundary_tensor_mb': tensor_mb(boundaries),
    })
    del boundaries

torch.cuda.empty_cache()
show_rows(boundary_info_rows)


,input,triangles,voxels,intersect_ms,intersect_peak_alloc_mb,brick_hash_slots,brick_bits_shape
0,original,280333,4676307,22.240225,381.709473,524288,"(262144, 16)"
1,subdivided,10304077,4676306,49.344479,1228.146484,524288,"(262144, 16)"


,input,faces,boundary_edges,extract_ms,boundary_tensor_mb
0,original,280333,227023,0.270336,5.196144
1,subdivided,10304077,730173,19.378176,16.712334


## Correctness Against CPU `boundry_qef`

Old/current GPU methods internally extract boundaries with the temporary CUDA hash-count extractor because their project functions take `boundaries`. Ref directly calls the real project `boundary_qef_ref(vertices, faces, ...)` and uses its own internal boundary extraction. CPU uses canonical boundary edges from the same mesh topology and the same `intersect_qef.voxels` rows. Outputs are canonicalized by voxel key before error metrics.

In [6]:
BOUNDARY_CANON = {}
correctness_rows = []

for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    cpu_allowed = input_name != 'subdivided' or RUN_CPU_SUBDIVIDED
    if not cpu_allowed:
        print(f'skip CPU boundary_qef for {input_name}; set FDG_RUN_CPU_SUBDIVIDED=1 to enable')
        continue

    print('running CPU boundary_qef for', input_name, 'faces:', case['faces_shape'][0], 'voxels:', int(inter.voxels.shape[0]))
    cpu_qef = boundary_qef_cpu_py(case, inter)
    cpu_can = canonical_boundary_qef(inter.voxels, cpu_qef)
    BOUNDARY_CANON[(input_name, 'cpu')] = cpu_can

    gpu_methods = {
        'old': boundary_qef_old_py,
        'ref': boundary_qef_ref_py,
        'new': boundary_qef_new_py,
    }
    for method, fn in gpu_methods.items():
        torch.cuda.synchronize()
        qef = fn(case, inter)
        torch.cuda.synchronize()
        can = canonical_boundary_qef(inter.voxels, qef)
        BOUNDARY_CANON[(input_name, method)] = can
        correctness_rows.append({'input': input_name, 'method': method, **boundary_error_metrics(cpu_can, can)})
        del qef

show_rows(correctness_rows)


running CPU boundary_qef for original faces: 280333 voxels: 4676307
running CPU boundary_qef for subdivided faces: 10304077 voxels: 4676306


,input,method,occ_equal,count,qef_max_abs,qef_rmse,qef_rel_l2,trace_max_abs,nonzero_cpu,nonzero_other
0,original,old,True,4676307,0.000008,3.322306e-08,5.327356e-08,0.000008,526343,526343
1,original,ref,True,4676307,0.000010,4.399057e-08,7.053415e-08,0.000011,526343,526343
2,original,new,True,4676307,0.000008,3.171687e-08,5.085928e-08,0.000013,526343,526343
3,subdivided,old,True,4676306,0.000008,3.658456e-08,5.415294e-08,0.000008,519823,519823
4,subdivided,ref,True,4676306,0.000011,4.635404e-08,6.861079e-08,0.000013,519823,519823
5,subdivided,new,True,4676306,0.000011,3.546825e-08,5.250156e-08,0.000011,519823,519823


## QEF Distribution Summary

This gives a compact view of how many rows received boundary QEF and the trace magnitude distribution for CPU/GPU methods.

In [7]:
dist_rows = []
for (input_name, method), can in BOUNDARY_CANON.items():
    q = can['qefs']
    trace = q[:, 0] + q[:, 4] + q[:, 7]
    active = q.abs().sum(dim=1) > 0
    active_trace = trace[active]
    dist_rows.append({
        'input': input_name,
        'method': method,
        'rows': int(q.shape[0]),
        'active_rows': int(active.sum().item()),
        'trace_sum': float(trace.sum().item()) if trace.numel() else 0.0,
        'trace_mean_active': float(active_trace.mean().item()) if active_trace.numel() else 0.0,
        'trace_max': float(trace.max().item()) if trace.numel() else 0.0,
    })
show_rows(dist_rows)


,input,method,rows,active_rows,trace_sum,trace_mean_active,trace_max
0,original,cpu,4676307,526343,3166076.0,6.015234,92.000008
1,original,old,4676307,526343,3166076.0,6.015234,92.000000
2,original,ref,4676307,526343,3166076.0,6.015234,92.000000
3,original,new,4676307,526343,3166076.0,6.015234,92.000008
4,subdivided,cpu,4676306,519823,3400924.0,6.542465,92.000008
5,subdivided,old,4676306,519823,3400924.0,6.542465,92.000000
6,subdivided,ref,4676306,519823,3400924.0,6.542465,92.000008
7,subdivided,new,4676306,519823,3400924.0,6.542465,92.000000


## GPU Timing And Memory

Cold runs clear the CUDA caching allocator before the call. Warm results report the median across repeats. Boundary extraction is part of each measured GPU function, matching the public Python signature used in this notebook.

In [8]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }


def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }


bench_rows = []
for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    methods = {
        'old': lambda case=case, inter=inter: boundary_qef_old_py(case, inter),
        'ref': lambda case=case, inter=inter: boundary_qef_ref_py(case, inter),
        'new': lambda case=case, inter=inter: boundary_qef_new_py(case, inter),
    }
    for method, fn in methods.items():
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        warm = summarize_records(warm_records)
        bench_rows.append({
            'input': input_name,
            'faces': int(case['faces_cuda_i32'].shape[0]),
            'voxels': int(inter.voxels.shape[0]),
            'mesh_mb': case['mesh_mb'],
            'method': method,
            'cold_ms': cold['ms'],
            'cold_peak_alloc_mb': cold['peak_alloc_mb'],
            'cold_peak_reserved_mb': cold['peak_reserved_mb'],
            **warm,
        })

show_rows(bench_rows)


,input,faces,voxels,mesh_mb,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,original,280333,4676307,6.275379,old,8.389536,527.934570,366.0,3.577344,3.470336,527.915039,0.0
1,original,280333,4676307,6.275379,ref,1.398784,217.109375,180.0,0.881312,0.867360,217.109375,0.0
2,original,280333,4676307,6.275379,new,2.818304,183.583496,182.0,1.012736,1.003808,183.583496,0.0
3,subdivided,10304077,4676306,159.220974,old,25.000959,784.712891,770.0,23.198880,22.003712,784.712891,0.0
4,subdivided,10304077,4676306,159.220974,ref,24.023041,1602.387207,1604.0,14.460928,13.264896,1602.387207,0.0
5,subdivided,10304077,4676306,159.220974,new,24.578943,784.712891,770.0,19.970896,19.666945,784.712891,0.0
